# Neural Network Foundations
**Summer of Science 2026 — CS03: Artificial Intelligence and Machine Learning**  
**Mohit Khyalia | IIT Bombay**

---

## Project Overview

Building a neural network from scratch in NumPy — connecting directly to the sigmoid, log-loss, and gradient descent already implemented in `logistic_regression_from_scratch.ipynb`.

### Main goals:

- Show that a single neuron with sigmoid activation is identical to Logistic Regression.
- Implement sigmoid, ReLU, and tanh activations and compare their gradients.
- Demonstrate why a single layer fails on XOR and a hidden layer fixes it.
- Implement a two-layer network forward and backward pass in NumPy.

---

## Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False})

## Single Neuron = Logistic Regression

A neuron computes `z = w·x + b`, then applies an activation function. With sigmoid, this is exactly the Logistic Regression model from Week 3.

In [2]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

# Single neuron forward pass
def neuron_forward(x, w, b):
    z = np.dot(w, x) + b
    return sigmoid(z), z

# Example: 3-feature input
np.random.seed(42)
x = np.array([0.5, -1.2, 0.8])
w = np.array([0.3, -0.5, 0.2])
b = 0.1

a, z = neuron_forward(x, w, b)
print(f'z (linear combination): {z:.4f}')
print(f'a (sigmoid output):     {a:.4f}  → interpreted as P(class=1)')

**Observation:**
The single neuron computes the same `z = w·x + b → sigmoid(z)` as `logistic_regression_from_scratch.ipynb`. The difference in a neural network is that multiple neurons are stacked in layers, each feeding its output into the next.

## Activation Functions

In [3]:
def relu(z):    return np.maximum(0, z)
def tanh(z):    return np.tanh(z)

z = np.linspace(-4, 4, 300)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, (name, fn, color) in zip(axes, [
    ('Sigmoid', sigmoid, '#1F3864'),
    ('ReLU',    relu,    '#C00000'),
    ('Tanh',    tanh,    '#2CA02C')
]):
    ax.plot(z, fn(z), color=color, lw=2)
    ax.axhline(0, color='k', lw=0.6, linestyle='--')
    ax.axvline(0, color='k', lw=0.6, linestyle='--')
    ax.set_title(name)
    ax.set_xlabel('z')

plt.tight_layout()
plt.show()

In [4]:
# Gradients
def sigmoid_grad(z): return sigmoid(z) * (1 - sigmoid(z))
def relu_grad(z):    return (z > 0).astype(float)
def tanh_grad(z):    return 1 - np.tanh(z) ** 2

fig, ax = plt.subplots(figsize=(8, 4))
for name, fn, color in [
    ('Sigmoid gradient', sigmoid_grad, '#1F3864'),
    ('ReLU gradient',    relu_grad,    '#C00000'),
    ('Tanh gradient',    tanh_grad,    '#2CA02C')
]:
    ax.plot(z, fn(z), label=name, color=color, lw=2)
ax.set_title('Activation Gradients')
ax.set_xlabel('z')
ax.set_ylabel('d(activation)/dz')
ax.legend()
plt.tight_layout()
plt.show()

**Observation:**
Sigmoid and tanh gradients shrink toward zero for large |z| — the vanishing gradient problem. When many layers multiply these small gradients together during backpropagation, early layers receive almost no update signal. ReLU avoids this: its gradient is either 0 (z < 0) or 1 (z > 0), keeping gradients from vanishing in deep networks.

## The XOR Problem

In [5]:
# XOR: no single straight line can separate the classes
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])  # XOR labels

fig, ax = plt.subplots(figsize=(5, 5))
colors = ['#1F3864' if label == 0 else '#C00000' for label in y_xor]
ax.scatter(X_xor[:, 0], X_xor[:, 1], c=colors, s=200, zorder=3)
for i, (xi, yi, label) in enumerate(zip(X_xor[:,0], X_xor[:,1], y_xor)):
    ax.annotate(f'XOR={label}', (xi, yi), textcoords="offset points",
                xytext=(8, 5), fontsize=11)
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 1.5)
ax.set_title('XOR — Not Linearly Separable')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
plt.tight_layout()
plt.show()

**Observation:**
No straight line separates the blue points (XOR=0) from the red points (XOR=1). A single neuron with any linear activation can only draw one hyperplane — it fails on XOR regardless of how the weights are trained. A hidden layer maps inputs to a new space where the classes become linearly separable.

## Two-Layer Network — Forward Pass

In [6]:
class TwoLayerNet:
    def __init__(self, n_in, n_hidden, n_out, seed=42):
        rng = np.random.default_rng(seed)
        # He initialisation for ReLU hidden layer
        self.W1 = rng.normal(0, np.sqrt(2/n_in), (n_hidden, n_in))
        self.b1 = np.zeros((n_hidden, 1))
        # Xavier initialisation for sigmoid output
        self.W2 = rng.normal(0, np.sqrt(1/n_hidden), (n_out, n_hidden))
        self.b2 = np.zeros((n_out, 1))

    def forward(self, X):
        # X shape: (n_features, m_samples)
        self.Z1 = self.W1 @ X + self.b1
        self.A1 = relu(self.Z1)
        self.Z2 = self.W2 @ self.A1 + self.b2
        self.A2 = sigmoid(self.Z2)
        return self.A2

net = TwoLayerNet(n_in=2, n_hidden=4, n_out=1)
X_xor_T = X_xor.T  # shape (2, 4)
output = net.forward(X_xor_T)
print('Network output (before training):')
for i in range(4):
    print(f'  Input {X_xor[i]} → {output[0,i]:.4f}  (true XOR = {y_xor[i]})')

## Backpropagation — Gradient Computation

In [7]:
def backward(net, X, y, m):
    """Compute gradients via backpropagation (chain rule)."""
    # Output layer gradient
    dZ2 = net.A2 - y                       # (1, m)
    dW2 = (1/m) * dZ2 @ net.A1.T           # (1, n_hidden)
    db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)

    # Hidden layer gradient (chain rule through ReLU)
    dA1 = net.W2.T @ dZ2                   # (n_hidden, m)
    dZ1 = dA1 * relu_grad(net.Z1)          # element-wise: ReLU gate
    dW1 = (1/m) * dZ1 @ X.T
    db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)

    return dW1, db1, dW2, db2

# Show gradient shapes
y_xor_T = y_xor.reshape(1, -1)
m = X_xor_T.shape[1]
grads = backward(net, X_xor_T, y_xor_T, m)
for name, g in zip(['dW1', 'db1', 'dW2', 'db2'], grads):
    print(f'{name}: shape {g.shape}')

## Training the Network on XOR

In [8]:
def train_xor(lr=0.5, n_iter=5000):
    net = TwoLayerNet(n_in=2, n_hidden=4, n_out=1, seed=0)
    X_T = X_xor.T
    y_T = y_xor.reshape(1, -1)
    m   = X_T.shape[1]
    losses = []

    for _ in range(n_iter):
        A2 = net.forward(X_T)
        A2_clip = np.clip(A2, 1e-12, 1 - 1e-12)
        loss = -np.mean(y_T * np.log(A2_clip) + (1 - y_T) * np.log(1 - A2_clip))
        losses.append(loss)

        dW1, db1, dW2, db2 = backward(net, X_T, y_T, m)
        net.W1 -= lr * dW1
        net.b1 -= lr * db1
        net.W2 -= lr * dW2
        net.b2 -= lr * db2

    return net, losses

net_trained, losses = train_xor()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(losses, color='#1F3864', lw=1.5)
ax.set_xlabel('Iteration')
ax.set_ylabel('Log-Loss')
ax.set_title('XOR — Two-Layer Network Training Loss')
plt.tight_layout()
plt.show()

In [9]:
# Final predictions
final_out = net_trained.forward(X_xor.T)
preds = (final_out[0] >= 0.5).astype(int)
print('Input   | True XOR | Predicted | Output')
for i in range(4):
    print(f'{X_xor[i]}  |    {y_xor[i]}     |     {preds[i]}     | {final_out[0,i]:.4f}')

**Observation:**
The two-layer network learns XOR perfectly — something impossible for a single neuron. The hidden layer creates intermediate representations that transform the input space so the output neuron can draw a straight boundary. The same backpropagation + gradient descent loop used in Week 3 drives all of this; the only addition is the chain rule applied across layers.